# 04 - Model Building
**Objective**: Select features, train 4 ML models, and compare performance.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from src.utils.helpers import print_header, load_dataframe
from src.utils.config import PROCESSED_DATA_DIR
from src.features.selection import select_features, get_feature_columns
from src.models.trainer import prepare_train_test, train_all_models, get_best_model, cross_validate_models
from src.visualization.plots import plot_feature_importance

## 4.1 Load Engineered Data

In [ ]:
df = load_dataframe(os.path.join(PROCESSED_DATA_DIR, 'df_engineered.csv'))
print(f"Shape: {df.shape}")
print(f"Churn rate: {df['is_churned'].mean():.1%}")

## 4.2 Feature Selection

In [ ]:
selected_features, importance = select_features(df)
print(f"\nSelected {len(selected_features)} features")

In [ ]:
plot_feature_importance(importance)

## 4.3 Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, scaler = \
    prepare_train_test(df, selected_features)

## 4.4 Train All Models

In [ ]:
results = train_all_models(
    X_train, X_test, y_train, y_test,
    X_train_scaled, X_test_scaled
)

## 4.5 Model Comparison (Holdout)

In [ ]:
from src.models.evaluator import create_comparison_table

comparison = create_comparison_table(results)
print("\nModel Comparison (Holdout Test Set):")
display(comparison.style.highlight_max(axis=0, color='lightgreen'))

best_name = get_best_model(results)

## 4.6 Cross-Validation (5-Fold Stratified)

Cross-validation ensures our results are **stable and not dependent on a single lucky train/test split**.
Each model is trained 5 times on different folds, and we report mean +/- std.

In [ ]:
cv_results = cross_validate_models(df, selected_features, n_folds=5)

print("\nCross-Validation Results (mean +/- std):")
display(cv_results)

### Interpretation
- **Low std** (< 0.01) = model is stable across different data splits ✅
- **CV matches holdout** = no overfitting to the test set ✅
- **High variance** between folds would suggest the model is sensitive to training data

## 4.7 Save Results for Next Notebooks

In [ ]:
import joblib

# Save selected features list and results for notebooks 05-06
joblib.dump({
    'selected_features': selected_features,
    'importance': importance,
    'results': {k: {kk: vv for kk, vv in v.items() if kk not in ['model']}
                for k, v in results.items()},
    'best_model_name': best_name,
    'scaler': scaler,
    'y_test': y_test,
    'cv_results': cv_results,
}, os.path.join(PROCESSED_DATA_DIR, 'training_artifacts.pkl'))
print("Saved training artifacts")